In [1]:
import tensorflow as tf
from tensorflow.keras import layers, Model
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

In [2]:
# Load dataset
data = pd.read_csv("augmented_hand_landmarks.csv")
X = data.drop(columns=['label']).values
y = data['label'].values

In [3]:
def create_pairs(X, y):
    pairs = []
    unique_labels = np.unique(y)
    label_to_indices = {label: np.where(y == label)[0] for label in unique_labels}
    
    for label in unique_labels:
        indices = label_to_indices[label]
        num_indices = len(indices)
        
        # Create positive pairs only
        for i in range(num_indices):
            # Ensure we don't pair a sample with itself
            possible_pairs = indices[indices != indices[i]]
            if len(possible_pairs) > 0:
                j = np.random.choice(possible_pairs)
                pairs.append([X[indices[i]], X[j]])
    
    return np.array(pairs)

pairs = create_pairs(X, y)
pairs_train, pairs_test = train_test_split(pairs, test_size=0.2, random_state=42)

MemoryError: Unable to allocate 536. MiB for an array with shape (585396, 2, 60) and data type float64

In [ ]:
def build_base_network(input_shape):
    input = layers.Input(shape=input_shape)
    
    # First dense layer
    x = layers.Dense(256, activation=None, kernel_initializer='he_normal')(input)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
   
    # Second dense layer
    x = layers.Dense(128, activation=None, kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    
    # Third dense layer
    x = layers.Dense(128, activation=None, kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    
    # Fourth dense layer
    x = layers.Dense(64, activation=None, kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    
    # Fifth dense layer
    x = layers.Dense(256, activation=None, kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    
    # Final embedding layer
    x = layers.Dense(128, activation=None)(x)
    
    return Model(input, x)

input_shape = pairs_train.shape[2:]
base_network = build_base_network(input_shape)

# Create Siamese network
input_a = layers.Input(shape=input_shape)
input_b = layers.Input(shape=input_shape)

embedding_a = base_network(input_a)
embedding_b = base_network(input_b)

# Concatenate embeddings for InfoNCE
concatenated = layers.Concatenate()([embedding_a, embedding_b])
siamese_model = Model(inputs=[input_a, input_b], outputs=concatenated)

In [ ]:
def info_nce_loss(temperature=0.1):
    def loss(y_true, y_pred):
        batch_size = tf.shape(y_pred)[0]
        
        # Split the concatenated embeddings
        z_i = y_pred[:, :128]  # anchor
        z_j = y_pred[:, 128:]  # positive
        
        # Normalize embeddings
        z_i = tf.math.l2_normalize(z_i, axis=1)
        z_j = tf.math.l2_normalize(z_j, axis=1)
        
        # Gather all embeddings to create negative pairs
        z_all = tf.concat([z_i, z_j], axis=0)
        
        # Compute similarity matrix
        sim = tf.matmul(z_i, tf.transpose(z_all)) / temperature
        
        # Create labels for positive pairs
        labels = tf.range(batch_size)
        masks = tf.one_hot(labels, batch_size * 2)
        
        # Compute InfoNCE loss
        loss = tf.keras.losses.sparse_categorical_crossentropy(
            labels, sim, from_logits=True)
        
        return tf.reduce_mean(loss)
    return loss

siamese_model.compile(
    optimizer='adam',
    loss=info_nce_loss(temperature=0.1),
    metrics=['accuracy']
)

In [ ]:
history = siamese_model.fit(
    [pairs_train[:, 0], pairs_train[:, 1]], 
    np.zeros((len(pairs_train),)),  # Dummy labels since InfoNCE doesn't use them
    validation_data=([pairs_test[:, 0], pairs_test[:, 1]], np.zeros((len(pairs_test),))),
    batch_size=256,
    epochs=100
)

In [ ]:
# Plot training history
plt.figure(figsize=(12, 5))

# Plot loss
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# Plot accuracy
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()